# ⚠ INVALID FOR PROMOTION — 本 notebook 为 invalid historical result

本卡原结论与指标已作废：隔夜信号的 `09:00` 一小时 bar 覆盖 `09:00–10:00`，其 close 是 10:00 价格，在 09:30 现金开盘之后（前视）。不安全 cutoff IC=+0.2200，安全 cutoff IC=−0.0119。
原输出仅保留用于审计；**不得用于晋升、实盘或任何可用策略声明**。详见 `integrity_correction.json` 与 `conclusion.md`。


## 只读完整性校验（不重新拟合、不择参、不调用已禁 run）

本 notebook 只做两件事：①只读加载已标记 `invalid_for_promotion` 的 summary；
②执行 `integrity_correction` 并离线校验源时间戳快照。所有原始表格/指标标注为 **invalid**。


In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'night_session_signal_20261010'
# Read-only load of the already-marked-invalid summary. The run/writer are
# unconditionally blocked and are deliberately NOT imported or called here.
summary = json.loads((CARD / 'summary.json').read_text())
print('decision:', summary['decision'])
print('promotion_eligible:', summary['promotion_eligible'])
print('objective_achieved:', summary['objective_achieved'])
print('integrity.status:', summary['integrity']['status'])


decision: invalid_for_promotion_look_ahead
promotion_eligible: False
objective_achieved: False
integrity.status: invalid_for_promotion


In [2]:
from src.analysis.night_session_integrity import integrity_correction
correction = integrity_correction(ROOT)
print('INVALID FOR PROMOTION:', correction['status'])
print('known_defects:')
for d in correction['known_defects']:
    print('  -', d)


INVALID FOR PROMOTION: invalid_for_promotion
known_defects:
  - look_ahead: 09:00 one-hour bar spans 09:00-10:00 but was selected by index.time < 09:30
  - return_convention_mismatch: headline open-to-close vs native open-to-next-open
  - turnover_undercharged: adjacent-day target difference, not intraday round trip
  - benchmark_mislabeled: 'SOXX buyhold' is an intraday session return
  - threshold_reselected_in_diagnostic: threshold chosen inside the evaluation window
  - recorder_incomplete: recorder stored only a summary yet claimed panels


In [3]:
import pandas as pd
span = correction['bar_span_audit']
print('span audit status:', span['status'], '| all_five_fields_identical:', span['all_five_fields_identical'])
print('snapshot sha256:', correction['span_source_snapshot_sha256'])
print('verified raw parquet sha256:', span.get('verified_raw_parquet_sha256'))
display(pd.DataFrame(span['days'])[['day','status','identical']])


span audit status: ok | all_five_fields_identical: True
snapshot sha256: 4a5f55b37fc3a0298420c32008e6940d6e79c7347bd8da64e546ffe634a97ae1
verified raw parquet sha256: 5caf8600ca1ad8428032ed8049bd64cca330faed22685d58c05ac1fa3b2b0dd6


,day,status,identical
0,2026-10-05,ok,True
1,2026-10-06,ok,True
2,2026-10-07,ok,True
3,2026-10-08,ok,True


In [4]:
# Forensic IC sensitivity only: NOT tradable, NOT new research.
print('FORENSIC timestamp-sensitivity (not tradable, not new Qlib research):')
print('  unsafe cutoff 09:30 IC:', correction['ic_unsafe_cutoff_09_30'])
print('  safe   cutoff 09:00 IC:', correction['ic_safe_cutoff_09_00'])


FORENSIC timestamp-sensitivity (not tradable, not new Qlib research):
  unsafe cutoff 09:30 IC: {'status': 'ok', 'cutoff': '09:30', 'sessions': 498, 'ic_soxl': 0.22004568778753475, 'ic_excess': 0.22439527374705595}
  safe   cutoff 09:00 IC: {'status': 'ok', 'cutoff': '09:00', 'sessions': 498, 'ic_soxl': -0.01191814227880693, 'ic_excess': -0.007243982756510589}


In [5]:
# Original historical metrics are preserved but INVALID; the same-account native
# reconciliation cannot be verified. Do not read them as usable performance.
print('superseded (INVALID) original metrics payload keys:')
print(' ', sorted(summary['integrity']['superseded_metrics_preserved'].keys()))
print('ORIGINAL-INVALID native_audit status field was:', repr(summary.get('native_audit', {}).get('status')))
print('=> that value is INVALID and unverifiable; same-account reconciliation NOT verifiable')


superseded (INVALID) original metrics payload keys:
  ['note', 'original_decision', 'original_promotion_eligible']
ORIGINAL-INVALID native_audit status field was: 'passed'
=> that value is INVALID and unverifiable; same-account reconciliation NOT verifiable


In [6]:
print('决定:', summary['decision'])
print('是否可晋升:', summary['promotion_eligible'])
print('任何修正研究须使用新的卡片身份；不得本卡重跑或宣称可用。')


决定: invalid_for_promotion_look_ahead
是否可晋升: False
任何修正研究须使用新的卡片身份；不得本卡重跑或宣称可用。


## 中文结论


In [7]:
from IPython.display import Markdown, display
display(Markdown((CARD/'conclusion.md').read_text()))


# 夜盘信号卡：完整性更正（INVALID FOR PROMOTION）

> **状态：`invalid_for_promotion_look_ahead`。本卡原结论与指标全部作废，不得用于晋升、实盘或任何可用策略声明。**
> 原指标保留为 **invalid historical result**，仅用于审计；原 qlib Recorder 保留不改。

## 缺陷（机器可验证）

1. **前视（决定性）**：隔夜信号的窗口用了 `index.time < 09:30`，但 Yahoo NQ 的 `09:00` 一小时 bar 覆盖 `09:00–10:00`，其 close 是 **10:00 价格**，在 09:30 现金开盘之后。已用 `interval=1h` 与 `interval=5m` 对齐核实 2026-10-05/06/07/08：`09:00` 一小时 bar 的 OHLCV 与 `09:00≤t<10:00` 的 5m 聚合**完全一致**。
   - 不安全 cutoff（09:30）：IC(SOXL) = **+0.2200**，IC(超额) = +0.2244
   - 安全 cutoff（09:00）：IC(SOXL) = **−0.0119**，IC(超额) = −0.0072
   - 即：+0.22 的“信号”完全来自前视 bar。
2. **回报口径不一致**：headline 是 open→close，native 对账用 open→次日 open。
3. **成本漏记**：换手按相邻日 target 差计，日内 round trip 未计足。
4. **基准误标**：所谓 “SOXX buyhold” 实为日内 session return。
5. **诊断内重选阈值**：阈值在各评估窗内被重新选择。
6. **Recorder 不完整**：只保存 summary，却声称有 panels。

## 更正内容（最小隔离，不重做有泄漏的 pipeline）

- `summary.json`：`decision` 改为 `invalid_for_promotion_look_ahead`，`promotion_eligible=false`，`objective_achieved=false`，并写入 `integrity` 区块；原指标保留在 `integrity.superseded_metrics_preserved`。
- `integrity_correction.json`：机器可验证的快照（bar span 审计 + 两个 cutoff 的 IC 复算）。
- `src/workflow/night_session_signal.py`：**无条件阻断**——`run_night_session_signal` 对任意目标目录（包括全新空目录）都抛 `blocked_known_integrity_defects`，不依赖是否已有 invalid 标记；legacy writer 同样拒绝写入任何 `promotion_eligible`/`objective_achieved` 或 `stable_*` 卡片。
- `src/analysis/night_session_integrity.py`：bar-end 可用性 helper（`bar_available_at` / `selectable_at`），供未来跨界 bar 未来扰动回归。

## 说明

- 两个 cutoff 的 IC 只是**源时间戳敏感性取证**，不是可交易结果，也不是新的 Qlib 研究；其余错误口径未修复，只隔离了时点。
- 后续任何修正研究必须使用**新的卡片身份**，不得在本卡上重跑或宣称成功。

## qlib Recorder（保留，不改）

- `6719efdbc2734e999feb282bcf7cd42e`（原始，invalid historical result）

> **再次声明：本卡 INVALID FOR PROMOTION，不得 eligible 或 objective achieved。**
